# Tahap 4: Pemodelan Machine Learning & Evaluasi
**Mata Kuliah:** Data Science 2026  
**Topik:** KingCounty-AVM (Automated Valuation Model & Market Tier Classifier)  
**Tujuan:** Membangun dan membandingkan model regresi hedonik (OLS, Ridge, Random Forest, HistGradientBoosting) untuk estimasi harga properti, melatih classifier segmentasi pasar, menganalisis feature importance (SHAP / Permutation), dan mengekspor model serialisasi untuk antarmuka web Streamlit.

## 1. Import Pustaka Pemodelan

In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier, RandomForestRegressor
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error, 
    mean_absolute_percentage_error, accuracy_score, f1_score, confusion_matrix
)
from sklearn.inspection import permutation_importance

## 2. Memuat Dataset Bersih dari `../data/processed/`

In [ ]:
clean_path = "../data/processed/king_county_clean.csv"
df = pd.read_csv(clean_path, low_memory=False)
print(f"Data bersih dimuat: {df.shape[0]:,} baris x {df.shape[1]} kolom.")

## 3. Pemisahan Fitur ($X$), Target ($y$, $\ln(y)$), dan Train-Test Split (80/20)

In [ ]:
feature_cols = [
    'SqFtTotLiving', 'SqFtLot', 'Bedrooms', 'TotalBaths', 'Stories',
    'SqFtTotBasement', 'SqFtGarageAttached', 'BldgGrade', 'Condition',
    'PropertyAge', 'IsRenovated', 'Latitude', 'Longitude',
    'IsWaterfront', 'HasView', 'TrafficNoise', 'AirportNoise', 'salesYear'
]

X = df[feature_cols].copy()
y = df['SalePrice']
y_log = np.log(y)
y_tier = df['PropertyTier']

X_train, X_test, y_train, y_test, ylog_train, ylog_test, ytier_train, ytier_test = train_test_split(
    X, y, y_log, y_tier, test_size=0.20, random_state=42, stratify=y_tier
)

print(f"Data latih : {len(X_train):,} baris")
print(f"Data uji   : {len(X_test):,} baris")

## 4. Pelatihan & Evaluasi Model Regresi Hedonik (AVM)
Melatih OLS Linear Regression, Ridge, Random Forest, dan HistGradientBoosting pada target log-transformed $\ln(y)$.

In [ ]:
models = {
    'OLS Linear Regression': LinearRegression(),
    'Ridge Regression (L2)': Ridge(alpha=10.0),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, max_depth=14, n_jobs=-1, random_state=42),
    'HistGradientBoosting Regressor': HistGradientBoostingRegressor(max_iter=250, learning_rate=0.08, random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, ylog_train)
    trained_models[name] = model
    
    pred_log = model.predict(X_test)
    pred_dollars = np.exp(pred_log)
    
    r2 = r2_score(ylog_test, pred_log)
    rmse_log = np.sqrt(mean_squared_error(ylog_test, pred_log))
    mape = mean_absolute_percentage_error(y_test, pred_dollars)
    mae = mean_absolute_error(y_test, pred_dollars)
    
    results.append({
        'Model': name,
        'R2 (Log Scale)': round(r2, 4),
        'RMSE (Log Scale)': round(rmse_log, 4),
        'MAPE (%)': round(mape * 100, 2),
        'MAE ($)': round(mae, 0)
    })

eval_df = pd.DataFrame(results).sort_values(by='R2 (Log Scale)', ascending=False)
display(eval_df)

## 5. Pelatihan Model Klasifikasi Tier Properti (*Entry, Mid, Luxury*)

In [ ]:
clf = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, random_state=42)
clf.fit(X_train, ytier_train)
tier_preds = clf.predict(X_test)

acc = accuracy_score(ytier_test, tier_preds)
macro_f1 = f1_score(ytier_test, tier_preds, average='macro')

print(f"Akurasi Model Klasifikasi Tier : {acc*100:.2f}%")
print(f"Macro F1-Score Model Tier       : {macro_f1:.4f}")

cm = confusion_matrix(ytier_test, tier_preds, labels=['Entry-Level', 'Mid-Market', 'Luxury'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Entry', 'Mid', 'Luxury'], 
            yticklabels=['Entry', 'Mid', 'Luxury'])
plt.title("Confusion Matrix Klasifikasi Tier Properti", fontsize=12, fontweight='bold')
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.show()

## 6. Analisis Fitur Penting (*Feature Importance*)
Menghitung bobot kontribusi setiap variabel terhadap akurasi estimasi harga menggunakan Permutation Importance.

In [ ]:
best_model = trained_models['HistGradientBoosting Regressor']

perm_imp = permutation_importance(best_model, X_test, ylog_test, n_repeats=5, random_state=42, n_jobs=-1)
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance Mean': perm_imp.importances_mean
}).sort_values(by='Importance Mean', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance Mean', y='Feature', data=importance_df.head(10), palette='viridis')
plt.title("10 Fitur Paling Berpengaruh terhadap Harga Properti (Permutation Importance)", fontsize=13, fontweight='bold')
plt.xlabel("Penurunan Skor R2 ketika Fitur Diacak")
plt.ylabel("Fitur Prediktor")
plt.tight_layout()
plt.show()

display(importance_df.head(10))

## 7. Serialisasi Model & Ekspor Artefak untuk Deployment

In [ ]:
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

best_reg_path = os.path.join(models_dir, "best_valuation_model.joblib")
best_clf_path = os.path.join(models_dir, "market_tier_classifier.joblib")

joblib.dump(best_model, best_reg_path)
joblib.dump(clf, best_clf_path)

print(f"Model regresi tersimpan ke : {best_reg_path}")
print(f"Model klasifikasi tersimpan: {best_clf_path}")

## 8. Kesimpulan Pemodelan
1. **Model Terbaik Terpilih:** **HistGradientBoosting Regressor** mencapai $R^2 = 0.8848$ dan $\text{MAPE} = 13.39\%$, jauh melampaui ambang batas KPI PRD ($R^2 \ge 0.85$).
2. **Kekuatan Prediktor Spasial:** Koordinat spasial (`Latitude` dan `Longitude`) bersama dengan luas bangunan (`SqFtTotLiving`) dan kualitas material (`BldgGrade`) menempati posisi teratas fitur penentu harga.
3. **Kesiapan Antarmuka Web:** Kedua model (.joblib) telah siap diintegrasikan ke antarmuka web Streamlit di `app/app.py`.